# 02 学習済みCNNへのexact TT-Conv接続

学習済み`FashionMNISTCNN`の`conv2`を、dense kernelを保持しないexact `TTConv2d`へ置換します。

validation全体について、**logitsの数値的一致**と**予測classの一致**を別々に確認します。rank制限による圧縮は、次のNotebookで扱います。


## 1. このNotebookで確認すること

1. TT coresをparameterとして保持する`TTConv2d`を作る。
2. 学習済みCNNの`conv2`だけをexact `TTConv2d`へ置換する。
3. 同じvalidation入力に対するconv2出力とモデル全体のlogitsを比較する。
4. 数値的一致、予測一致、正解ラベルに対するaccuracyを分けて記録する。
5. 元モデルと置換対象外のstateが変更されていないことを確認する。

### このNotebookでは行わないこと

- rank制限、rank sweep、tensorization比較
- fine-tuning、optimizerによる複数stepの比較
- test setによる評価
- latency、MACs、peak memoryの測定
- `groups != 1`またはzero以外のpadding modeへの対応
- 実行前からの結果考察


## 2. importと評価条件

初期検証条件をCPU・float32に固定します。モデルは`eval()`、評価処理は`torch.inference_mode()`を使用します。parameter作成は勾配を無効化した評価範囲の外で行います。


In [28]:
from __future__ import annotations
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import copy
import math
import sys
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "nn_compression").is_dir():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        project_root = candidate
        break
else:
    raise RuntimeError("src/nn_compressionが見つかりません。")

from nn_compression.compression import (
    dense_to_tt_matrix_cores,
    tt_linear_forward,
    tt_matrix_num_parameters,
    tt_matrix_ranks,
)
from nn_compression.datasets import (
    get_fashion_mnist_datasets,
    split_fashion_mnist_dataset,
)
from nn_compression.metrics import (
    agreement,
    count_parameters,
    logits_rmse,
)
from nn_compression.models import FashionMNISTCNN
from nn_compression.training import evaluate
from nn_compression.utils import set_seed

SEED = 0
DEVICE = torch.device("cpu")
DTYPE = torch.float32
VALIDATION_BATCH_SIZE = 32

CONV2_RMSE_TOLERANCE = 1e-5
LOGITS_RMSE_TOLERANCE = 1e-5
MAX_ABS_LOGITS_DIFFERENCE_TOLERANCE = 1e-4

OUT_MODES = (4, 4, 4)
IN_MODES = (6, 6, 8)

set_seed(SEED)
print("project root:", project_root)
print("device / dtype:", DEVICE, DTYPE)


project root: D:\dev\nn-compression-svd-dmrg
device / dtype: cpu torch.float32


## 3. baseline checkpointとvalidation data

baselineの学習時と同じseedでFashion-MNISTの60,000件を55,000件のtrainと5,000件のvalidationへ分割します。test datasetは取得関数の戻り値として読み込みますが、このNotebookではloaderを作らず、評価にも使いません。


In [29]:
checkpoint_path = (
    project_root
    / "models"
    / "10_svd"
    / "30_fashion_mnist_cnn"
    / "01_cnn_baseline"
    / "fashion_mnist_cnn_baseline.pt"
)
data_dir = project_root / "data"

if not checkpoint_path.is_file():
    raise FileNotFoundError(f"baseline checkpointが見つかりません: {checkpoint_path}")

checkpoint = torch.load(
    checkpoint_path, map_location=DEVICE, weights_only=True
)
if "model_state_dict" not in checkpoint:
    raise KeyError("checkpointにmodel_state_dictがありません。")

dense_model = FashionMNISTCNN().to(device=DEVICE, dtype=DTYPE)
dense_model.load_state_dict(checkpoint["model_state_dict"])
dense_model.eval()

full_train_dataset, test_dataset = get_fashion_mnist_datasets(
    data_dir, download=True
)
train_dataset, validation_dataset = split_fashion_mnist_dataset(
    full_train_dataset, (55_000, 5_000), seed=SEED
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
)

assert len(validation_dataset) == 5_000
assert len(test_dataset) == 10_000
assert count_parameters(dense_model) == 421_642
assert checkpoint["model_parameters"] == 421_642
print("checkpoint:", checkpoint_path)
print("validation size:", len(validation_dataset))
print("test size (not evaluated):", len(test_dataset))


checkpoint: D:\dev\nn-compression-svd-dmrg\models\10_svd\30_fashion_mnist_cnn\01_cnn_baseline\fashion_mnist_cnn_baseline.pt
validation size: 5000
test size (not evaluated): 10000


## 4. conv2をexact TT-matrixへ変換する

`conv2.weight`のshapeは`(64, 32, 3, 3)`です。これを出力channelと1受容野の入力を結ぶ`(64, 288)`行列として扱います。

$$
(m_1,m_2,m_3)=(4,4,4),
\qquad
(n_1,n_2,n_3)=(6,6,8)
$$

を使用します。ここでは打ち切りを行いません。このtensorizationが最適だとは主張しません。


In [30]:
dense_conv2 = dense_model.conv2
assert tuple(dense_conv2.weight.shape) == (64, 32, 3, 3)
assert dense_conv2.groups == 1
assert dense_conv2.padding_mode == "zeros"

conv2_weight_matrix = dense_conv2.weight.detach().reshape(64, 288)
exact_cores = dense_to_tt_matrix_cores(
    conv2_weight_matrix,
    OUT_MODES,
    IN_MODES,
    max_rank=None,
)
exact_actual_ranks = tt_matrix_ranks(exact_cores)
exact_tt_core_parameter_count = tt_matrix_num_parameters(exact_cores)

assert exact_actual_ranks == (1, 24, 32, 1)
assert dense_conv2.weight.numel() == 18_432
assert exact_tt_core_parameter_count == 20_032
assert all(core.dtype == DTYPE for core in exact_cores)
assert all(core.device == DEVICE for core in exact_cores)

print("conv2 weight matrix:", tuple(conv2_weight_matrix.shape))
print("exact actual ranks:", exact_actual_ranks)
print("dense weight parameters:", dense_conv2.weight.numel())
print("exact TT core parameters:", exact_tt_core_parameter_count)


conv2 weight matrix: (64, 288)
exact actual ranks: (1, 24, 32, 1)
dense weight parameters: 18432
exact TT core parameters: 20032


## 5. TODO 1：`TTConv2d`を実装する

### 目的

TT coresを学習可能parameterとして保持し、dense kernelを作らずに2次元畳み込みと同じshapeの出力を返す`nn.Module`を作ります。

### 初期化時に受け取る値と意味

- `cores: list[torch.Tensor]`：TT-matrix cores。各coreの軸は左bond、出力mode、入力mode、右bond。
- `bias: torch.Tensor | None`：各出力channelへ加える値。存在する場合のshapeは`(out_channels,)`。
- `in_channels: int`：入力feature mapのchannel数。
- `out_channels: int`：出力feature mapのchannel数。
- `kernel_size: tuple[int, int]`：kernelの高さと幅。
- `stride: tuple[int, int]`：出力位置間の移動幅。
- `padding: tuple[int, int]`：入力の高さ・幅方向に加えるzero padding。
- `dilation: tuple[int, int]`：kernel要素間の間隔。
- `groups: int`：channel group数。このNotebookでは`1`のみ。
- `padding_mode: str`：padding方法。このNotebookでは`"zeros"`のみ。
- `out_modes: tuple[int, ...]`：出力channelを表すTTのphysical modes。
- `in_modes: tuple[int, ...]`：入力channelと空間kernelを表すTTのphysical modes。

### `forward`の入力と戻り値

- 入力`x: torch.Tensor`：shape `(batch_size, in_channels, input_height, input_width)`。
- 戻り値`torch.Tensor`：shape `(batch_size, out_channels, output_height, output_width)`。
- 戻り値は、各出力位置の入力patchへTT-matrixを作用させ、元の空間配置へ戻したfeature map。

### 成立条件

- `len(cores) == len(out_modes) == len(in_modes)`。
- 全coreが4階Tensorで、隣接bond次元が一致する。
- 左端と右端の外側bond次元が1。
- `math.prod(out_modes) == out_channels`。
- `math.prod(in_modes) == in_channels * kernel_height * kernel_width`。
- coresとbiasのdtype・deviceが一致する。
- coresは`nn.ParameterList`、biasは`nn.Parameter`または`None`として登録する。
- denseな`weight` parameterを保持しない。
- 通常forwardでdense kernelを再構成しない。

実装手順、軸の移動順、完成した関数呼び出しは示しません。


In [31]:
import math

import torch
from torch import nn


def _as_pair(value, name: str, minimum: int) -> tuple[int, int]:
    if isinstance(value, int) and not isinstance(value, bool):
        value = (value, value)

    if not isinstance(value, tuple) or len(value) != 2:
        raise TypeError(f"{name} must be an int or a tuple of two ints.")

    if not all(
        isinstance(v, int) and not isinstance(v, bool)
        for v in value
    ):
        raise TypeError(f"{name} must contain integers.")

    if any(v < minimum for v in value):
        raise ValueError(f"Each value in {name} must be >= {minimum}.")

    return value


def _validate_ttconv_inputs(
    cores,
    bias,
    in_channels,
    out_channels,
    groups,
    padding_mode,
    kernel_size,
    out_modes,
    in_modes,
):
    for name, value in (
        ("in_channels", in_channels),
        ("out_channels", out_channels),
        ("groups", groups),
    ):
        if (
            not isinstance(value, int)
            or isinstance(value, bool)
            or value <= 0
        ):
            raise ValueError(f"{name} must be a positive integer.")

    if groups != 1:
        raise NotImplementedError("Only groups=1 is currently supported.")

    if padding_mode != "zeros":
        raise NotImplementedError("Only zero padding is currently supported.")

    if not isinstance(cores, (list, tuple)) or len(cores) == 0:
        raise ValueError("cores must be a non-empty list or tuple.")

    for name, modes in (
        ("out_modes", out_modes),
        ("in_modes", in_modes),
    ):
        if not isinstance(modes, (list, tuple)) or len(modes) == 0:
            raise ValueError(f"{name} must be a non-empty list or tuple.")

        if not all(
            isinstance(v, int) and not isinstance(v, bool) and v > 0
            for v in modes
        ):
            raise ValueError(f"{name} must contain positive integers.")

    if not (len(cores) == len(out_modes) == len(in_modes)):
        raise ValueError("The numbers of cores and modes must match.")

    for k, core in enumerate(cores):
        if not isinstance(core, torch.Tensor):
            raise TypeError(f"Core {k} must be a Tensor.")

        if not torch.is_floating_point(core):
            raise TypeError(f"Core {k} must be a real floating-point Tensor.")

        if core.ndim != 4 or any(size <= 0 for size in core.shape):
            raise ValueError(f"Core {k} must have four positive dimensions.")

        if core.shape[1:3] != (out_modes[k], in_modes[k]):
            raise ValueError(f"Core {k} has incompatible physical modes.")

        if core.dtype != cores[0].dtype or core.device != cores[0].device:
            raise ValueError("All cores must have the same dtype and device.")

    if cores[0].shape[0] != 1 or cores[-1].shape[3] != 1:
        raise ValueError("The boundary ranks must be 1.")

    for left, right in zip(cores[:-1], cores[1:]):
        if left.shape[3] != right.shape[0]:
            raise ValueError("Adjacent bond dimensions must match.")

    if math.prod(out_modes) != out_channels:
        raise ValueError("The product of out_modes must equal out_channels.")

    if math.prod(in_modes) != in_channels * math.prod(kernel_size):
        raise ValueError(
            "The product of in_modes must equal "
            "in_channels * kernel_height * kernel_width."
        )

    if bias is not None:
        if not isinstance(bias, torch.Tensor):
            raise TypeError("bias must be a Tensor or None.")

        if tuple(bias.shape) != (out_channels,):
            raise ValueError("bias must have shape (out_channels,).")

        if bias.dtype != cores[0].dtype or bias.device != cores[0].device:
            raise ValueError("bias and cores must have the same dtype and device.")

In [32]:
class TTConv2d(nn.Module):
    def __init__(
        self,
        cores: list[torch.Tensor],
        bias: torch.Tensor | None,
        in_channels: int,
        out_channels: int,
        kernel_size: int | tuple[int, int],
        stride: int | tuple[int, int],
        padding: int | tuple[int, int],
        dilation: int | tuple[int, int],
        groups: int,
        padding_mode: str,
        out_modes: tuple[int, ...],
        in_modes: tuple[int, ...],
    ):
        # ① nn.Moduleの初期化
        super().__init__()

        # 設定の正規化と検証：完成済み
        kernel_size = _as_pair(kernel_size, "kernel_size", minimum=1)
        stride = _as_pair(stride, "stride", minimum=1)
        padding = _as_pair(padding, "padding", minimum=0)
        dilation = _as_pair(dilation, "dilation", minimum=1)

        _validate_ttconv_inputs(
            cores=cores,
            bias=bias,
            in_channels=in_channels,
            out_channels=out_channels,
            groups=groups,
            padding_mode=padding_mode,
            kernel_size=kernel_size,
            out_modes=out_modes,
            in_modes=in_modes,
        )

        # ② coresを学習可能parameterとして登録
        # 元のTensorと計算履歴・データ領域を共有しないコピーを使う
        self.cores = nn.ParameterList([
            nn.Parameter(core.detach().clone())
            for core in cores
        ])

        # biasも独立した学習可能parameterとして登録
        if bias is not None:
            self.bias = nn.Parameter(bias.detach().clone())
        else:
            self.register_parameter("bias",None)

        # ③ forwardで使う設定を保持
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.dilation = dilation

        self.groups = groups
        self.padding_mode = padding_mode

        self.out_modes = tuple(out_modes)
        self.in_modes = tuple(in_modes)

    # forwardは次に追加する
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 入力検証：完成済み
        if x.ndim != 4:
            raise ValueError(
                "x must have shape (batch, channels, height, width)."
            )

        if x.shape[1] != self.in_channels:
            raise ValueError("The input channel count does not match.")

        if x.dtype != self.cores[0].dtype:
            raise ValueError("x and cores must have the same dtype.")

        if x.device != self.cores[0].device:
            raise ValueError("x and cores must be on the same device.")

        batch_size, _, input_height, input_width = x.shape

        # 出力サイズ：完成済み
        output_height = (
            input_height
            + 2 * self.padding[0]
            - self.dilation[0] * (self.kernel_size[0] - 1)
            - 1
        ) // self.stride[0] + 1

        output_width = (
            input_width
            + 2 * self.padding[1]
            - self.dilation[1] * (self.kernel_size[1] - 1)
            - 1
        ) // self.stride[1] + 1

        if output_height <= 0 or output_width <= 0:
            raise ValueError(
                "The convolution settings produce a non-positive output size."
            )

        num_output_positions = output_height * output_width
        patch_width = self.in_channels * math.prod(self.kernel_size)

        # patchの取り出し：完成済み
        unfolded_patches = F.unfold(
            x,
            kernel_size=self.kernel_size,
            dilation=self.dilation,
            padding=self.padding,
            stride=self.stride,
        )

        # TODO A：各patchを1行にする
        # 入力 : (batch_size, patch_width, num_output_positions)
        # 出力 : (batch_size * num_output_positions, patch_width)
        # 行の順番：画像0の全位置 → 画像1の全位置 → …
        patch_matrix = unfolded_patches.permute(0,2,1).reshape(batch_size * num_output_positions, patch_width)

        # TODO B：この層のTT coresを使って、各patchの出力を求める
        # 入力 : (batch_size * num_output_positions, patch_width)
        # 出力 : (batch_size * num_output_positions, self.out_channels)
        patch_output = tt_linear_forward(
            patch_matrix,
            list(self.cores),
            self.out_modes,
            self.in_modes,
            bias=self.bias,
        )

        # TODO C：各行を画像・空間位置へ戻し、Conv形式にする
        # 出力 : (batch_size, self.out_channels,
        #         output_height, output_width)
        # (B*H*W, C_out) → (B, C_out, H, W)
        output = patch_output.reshape(
            batch_size, output_height, output_width, self.out_channels
        ).permute(0, 3, 1, 2)

        return output

In [33]:
if "TTConv2d" not in globals():
    print("TODO 1未実装: TTConv2d")
else:
    assert isinstance(TTConv2d, type)
    assert issubclass(TTConv2d, nn.Module)
    print("TODO 1 class definition: OK")


TODO 1 class definition: OK


## 6. TODO 2：学習済みCNNのconv2を置換する

### 入力変数と意味

- `dense_model: FashionMNISTCNN`：baseline checkpointを読み込んだ元モデル。
- `dense_conv2: nn.Conv2d`：置換元の`conv2`。
- `exact_cores: list[torch.Tensor]`：打ち切りなしで得たTT cores。
- `OUT_MODES`と`IN_MODES`：今回固定したtensorization。

### 作成する出力変数と意味

- `tt_conv2: TTConv2d`：exact coresと元bias・畳み込み設定を保持するTT層。
- `tt_model: nn.Module`：`conv2`だけを`tt_conv2`へ置換した比較モデル。

### 期待する状態

- `tt_model`と`dense_model`は別のモデルオブジェクト。
- `dense_model.conv2`は元の`nn.Conv2d`のまま。
- denseモデルのparameter・buffer値は置換前から変化しない。
- `tt_model`の`conv1`、`fc1`、`fc2`など、`conv2`以外のstateは元モデルと一致する。
- `tt_model.conv2`のactual rankは`(1, 24, 32, 1)`。
- `tt_model.conv2`はdenseな`weight`を保持しない。
- biasはshape`(64,)`の学習可能parameter。
- `tt_model`全体のparameter数は`423,242`。

コピー方法、置換式、constructor呼び出しの完成形は示しません。


In [34]:
dense_state_before = {
    name: value.detach().clone()
    for name, value in dense_model.state_dict().items()
}


In [35]:
# TODO 2
# ① exact coresと、元conv2の設定からTT層を作る：完成済み
tt_conv2 = TTConv2d(
    cores=exact_cores,
    bias=dense_conv2.bias,
    in_channels=dense_conv2.in_channels,
    out_channels=dense_conv2.out_channels,
    kernel_size=dense_conv2.kernel_size,
    stride=dense_conv2.stride,
    padding=dense_conv2.padding,
    dilation=dense_conv2.dilation,
    groups=dense_conv2.groups,
    padding_mode=dense_conv2.padding_mode,
    out_modes=OUT_MODES,
    in_modes=IN_MODES,
)

# ② 元モデルを変更しないよう、独立した比較用モデルを作る
tt_model = copy.deepcopy(dense_model)

# ③ 比較用モデルのconv2だけを、作成したTT層へ置き換える
tt_model.conv2 = tt_conv2

# 評価モードにする：完成済み
tt_model.eval()

FashionMNISTCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): TTConv2d(
    (cores): ParameterList(
        (0): Parameter containing: [torch.float32 of size 1x4x6x24]
        (1): Parameter containing: [torch.float32 of size 24x4x6x32]
        (2): Parameter containing: [torch.float32 of size 32x4x8x1]
    )
  )
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (relu3): ReLU()
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [36]:
todo2_names = ("tt_conv2", "tt_model")
todo2_missing = [name for name in todo2_names if name not in globals()]
if todo2_missing:
    print("TODO 2未実装:", todo2_missing)
else:
    assert tt_model is not dense_model
    assert isinstance(dense_model.conv2, nn.Conv2d)
    assert isinstance(tt_model.conv2, TTConv2d)
    assert tt_model.conv2 is tt_conv2
    assert isinstance(tt_model.conv2.cores, nn.ParameterList)
    assert not hasattr(tt_model.conv2, "weight")
    assert isinstance(tt_model.conv2.bias, nn.Parameter)
    assert tuple(tt_model.conv2.bias.shape) == (64,)
    assert tt_model.conv2.bias.requires_grad
    assert all(core.requires_grad for core in tt_model.conv2.cores)
    assert all(core.dtype == DTYPE for core in tt_model.conv2.cores)
    assert all(core.device == DEVICE for core in tt_model.conv2.cores)
    assert tt_model.conv2.bias.dtype == DTYPE
    assert tt_model.conv2.bias.device == DEVICE
    assert tt_matrix_ranks(list(tt_model.conv2.cores)) == exact_actual_ranks

    dense_state_after = dense_model.state_dict()
    assert dense_state_after.keys() == dense_state_before.keys()
    for name, before_value in dense_state_before.items():
        torch.testing.assert_close(dense_state_after[name], before_value)

    tt_state = tt_model.state_dict()
    for name, dense_value in dense_state_before.items():
        if not name.startswith("conv2."):
            assert name in tt_state
            torch.testing.assert_close(tt_state[name], dense_value)

    dense_model_parameter_count = count_parameters(dense_model)
    tt_model_parameter_count = count_parameters(tt_model)
    assert dense_model_parameter_count == 421_642
    assert tt_model_parameter_count == 423_242

    parameter_summary = pd.DataFrame(
        [
            {
                "model": "dense baseline",
                "total_parameters": dense_model_parameter_count,
                "conv2_weight_or_cores": dense_conv2.weight.numel(),
            },
            {
                "model": "exact TT-Conv",
                "total_parameters": tt_model_parameter_count,
                "conv2_weight_or_cores": exact_tt_core_parameter_count,
            },
        ]
    )
    dense_model.eval()
    tt_model.eval()
    display(parameter_summary)
    print("TODO 2 structural checks: OK")


,model,total_parameters,conv2_weight_or_cores
0,dense baseline,421642,18432
1,exact TT-Conv,423242,20032


TODO 2 structural checks: OK


## 7. validation全体で集計する指標

以下は5,000件のvalidation全体で集計します。batchごとのRMSEを単純平均しません。

### `conv2_output_rmse: float`

元CNNで`conv2`直前まで処理して得た同一feature mapを、dense `conv2`とexact `TTConv2d`へ入力したときの出力RMSEです。層単体の数値的一致を表し、受け入れ基準は`1e-5`以下です。

### `model_logits_rmse: float`

元モデルとTT置換モデルの全sample・全class logitsに対するRMSEです。今回のvalidation入力でモデル全体の数値差が許容範囲に収まるかを表し、受け入れ基準は`1e-5`以下です。

### `max_abs_logits_difference: float`

validation全体で観測されたlogits差の最大絶対値です。RMSEに局所的な大差が隠れていないかを確認し、受け入れ基準は`1e-4`以下です。

### `prediction_agreement: float`

元モデルとTT置換モデルが同じclassを予測したsampleの割合です。正解ラベルとの一致率ではありません。期待値は`1.0`です。

### `dense_validation_accuracy`と`tt_validation_accuracy`

それぞれのモデルが正解ラベルと一致したsampleの割合です。agreementとは別に表示します。agreementが`1.0`なら両accuracyも一致しますが、accuracyが同じだけでは両モデルの予測classが同じとは限りません。


In [37]:
def collect_exact_ttconv_diagnostics(
    dense_model: nn.Module,
    tt_model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
) -> dict[str, float]:
    """validation全体のconv2出力RMSEと最大logits差を返す。

    Args:
        dense_model: 元のFashionMNISTCNN。
        tt_model: conv2だけをexact TTConv2dへ置換したモデル。
        data_loader: shuffle=Falseのvalidation DataLoader。
        device: 両モデルと入力を配置するdevice。

    Returns:
        conv2_output_rmse:
            同じconv2直前feature mapに対する2層の全要素RMSE。
        max_abs_logits_difference:
            validation全体におけるモデルlogits差の最大絶対値。
    """
    dense_model.eval()
    tt_model.eval()

    conv2_squared_error_sum = 0.0
    conv2_element_count = 0
    max_abs_logits_difference = 0.0
    sample_count = 0

    with torch.inference_mode():
        for images, _ in data_loader:
            images = images.to(device=device, dtype=DTYPE)

            shared_conv2_input = dense_model.pool1(
                dense_model.relu1(dense_model.conv1(images))
            )
            dense_conv2_output = dense_model.conv2(shared_conv2_input)
            tt_conv2_output = tt_model.conv2(shared_conv2_input)
            conv2_difference = dense_conv2_output - tt_conv2_output

            conv2_squared_error_sum += (
                conv2_difference.double().square().sum().item()
            )
            conv2_element_count += conv2_difference.numel()

            dense_logits = dense_model(images)
            tt_logits = tt_model(images)
            batch_max_difference = (
                (dense_logits - tt_logits).abs().max().item()
            )
            max_abs_logits_difference = max(
                max_abs_logits_difference, batch_max_difference
            )
            sample_count += images.shape[0]

    if conv2_element_count == 0 or sample_count == 0:
        raise ValueError("空のDataLoaderでは診断指標を計算できません。")

    return {
        "conv2_output_rmse": (
            conv2_squared_error_sum / conv2_element_count
        ) ** 0.5,
        "max_abs_logits_difference": max_abs_logits_difference,
    }


def collect_prediction_disagreements(
    dense_model: nn.Module,
    tt_model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
) -> pd.DataFrame:
    """予測classが異なるsampleのlogits差とdense側marginを返す。

    Args:
        dense_model: 元のFashionMNISTCNN。
        tt_model: conv2だけをexact TTConv2dへ置換したモデル。
        data_loader: shuffle=Falseのvalidation DataLoader。
        device: 両モデルと入力を配置するdevice。

    Returns:
        不一致sampleごとのvalidation内index、正解label、両予測class、
        denseモデルの上位2class間margin、sample内最大logits差を持つDataFrame。
        不一致がなければ0行。
    """
    dense_model.eval()
    tt_model.eval()
    records: list[dict[str, int | float]] = []
    sample_offset = 0

    with torch.inference_mode():
        for images, labels in data_loader:
            images = images.to(device=device, dtype=DTYPE)
            labels = labels.to(device)
            dense_logits = dense_model(images)
            tt_logits = tt_model(images)
            dense_predictions = dense_logits.argmax(dim=1)
            tt_predictions = tt_logits.argmax(dim=1)
            mismatch_indices = torch.nonzero(
                dense_predictions != tt_predictions, as_tuple=False
            ).flatten()

            for batch_index in mismatch_indices.tolist():
                top2_values = torch.topk(
                    dense_logits[batch_index], k=2
                ).values
                records.append(
                    {
                        "validation_index": sample_offset + batch_index,
                        "label": int(labels[batch_index]),
                        "dense_prediction": int(dense_predictions[batch_index]),
                        "tt_prediction": int(tt_predictions[batch_index]),
                        "dense_top2_margin": float(
                            top2_values[0] - top2_values[1]
                        ),
                        "sample_max_abs_logits_difference": float(
                            (dense_logits[batch_index] - tt_logits[batch_index])
                            .abs()
                            .max()
                        ),
                    }
                )
            sample_offset += images.shape[0]

    return pd.DataFrame(
        records,
        columns=[
            "validation_index",
            "label",
            "dense_prediction",
            "tt_prediction",
            "dense_top2_margin",
            "sample_max_abs_logits_difference",
        ],
    )


In [38]:
if "tt_model" not in globals():
    print("TODO 2完了後にvalidation評価を実行できます。")
else:
    criterion = nn.CrossEntropyLoss()
    dense_validation_loss, dense_validation_accuracy = evaluate(
        dense_model, validation_loader, criterion, DEVICE
    )
    tt_validation_loss, tt_validation_accuracy = evaluate(
        tt_model, validation_loader, criterion, DEVICE
    )
    prediction_agreement = agreement(
        dense_model, tt_model, validation_loader, DEVICE
    )
    model_logits_rmse = logits_rmse(
        dense_model, tt_model, validation_loader, DEVICE
    )
    diagnostics = collect_exact_ttconv_diagnostics(
        dense_model, tt_model, validation_loader, DEVICE
    )
    conv2_output_rmse = diagnostics["conv2_output_rmse"]
    max_abs_logits_difference = diagnostics[
        "max_abs_logits_difference"
    ]

    evaluation_summary = pd.DataFrame(
        [
            {
                "model": "dense baseline",
                "validation_loss": dense_validation_loss,
                "validation_accuracy": dense_validation_accuracy,
            },
            {
                "model": "exact TT-Conv",
                "validation_loss": tt_validation_loss,
                "validation_accuracy": tt_validation_accuracy,
            },
        ]
    )
    equivalence_summary = pd.DataFrame(
        [
            {
                "metric": "conv2 output RMSE",
                "value": conv2_output_rmse,
                "acceptance_threshold": CONV2_RMSE_TOLERANCE,
            },
            {
                "metric": "model logits RMSE",
                "value": model_logits_rmse,
                "acceptance_threshold": LOGITS_RMSE_TOLERANCE,
            },
            {
                "metric": "max abs logits difference",
                "value": max_abs_logits_difference,
                "acceptance_threshold": MAX_ABS_LOGITS_DIFFERENCE_TOLERANCE,
            },
            {
                "metric": "prediction agreement",
                "value": prediction_agreement,
                "acceptance_threshold": 1.0,
            },
        ]
    )

    display(evaluation_summary)
    display(equivalence_summary)

    if prediction_agreement < 1.0:
        disagreement_diagnostics = collect_prediction_disagreements(
            dense_model, tt_model, validation_loader, DEVICE
        )
        display(disagreement_diagnostics)
    else:
        disagreement_diagnostics = pd.DataFrame()

    assert conv2_output_rmse <= CONV2_RMSE_TOLERANCE
    assert model_logits_rmse <= LOGITS_RMSE_TOLERANCE
    assert (
        max_abs_logits_difference
        <= MAX_ABS_LOGITS_DIFFERENCE_TOLERANCE
    )
    assert prediction_agreement == 1.0
    assert dense_validation_accuracy == tt_validation_accuracy
    print("validation-wide acceptance criteria: OK")


,model,validation_loss,validation_accuracy
0,dense baseline,0.223251,0.9194
1,exact TT-Conv,0.223251,0.9194


,metric,value,acceptance_threshold
0,conv2 output RMSE,5.581125e-07,0.00001
1,model logits RMSE,3.266623e-06,0.00001
2,max abs logits difference,1.239777e-05,0.00010
3,prediction agreement,1.000000e+00,1.00000


validation-wide acceptance criteria: OK


## 8. 実装後の確認項目

- [ ] `TTConv2d`はTT coresとbiasをparameterとして登録した
- [ ] `TTConv2d`はdenseなweightを保持していない
- [ ] `groups=1`とzero paddingの対応範囲を明示した
- [ ] 元のdenseモデルを変更せず、conv2以外のstateを維持した
- [ ] exact TTモデルのactual rankとparameter数を確認した
- [ ] conv2出力RMSEをvalidation全要素から集計した
- [ ] logits RMSEと最大logits差をvalidation全体で確認した
- [ ] prediction agreementとaccuracyを別々に表示した
- [ ] test setを評価していない

このNotebookはexact TT-Convを学習済みCNNへ正しく接続できることを確認した時点で終了します。rankを制限した圧縮とrank sweepは03で扱います。
